In [1]:
!git clone https://github.com/beatrizalvess6/deep_learning_project.git

Cloning into 'deep_learning_project'...
remote: Enumerating objects: 60864, done.
remote: Counting objects: 100% (831/831), done.
remote: Compressing objects: 100% (812/812), done.
remote: Total 60864 (delta 21), reused 828 (delta 19), pack-reused 60033 (from 1)
Receiving objects: 100% (60864/60864), 159.80 MiB | 15.08 MiB/s, done.
Resolving deltas: 100% (507/507), done.
Updating files: 100% (60012/60012), done.


In [2]:
%cd deep_learning_project

/content/deep_learning_project


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf

import keras
from keras import layers

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score

In [20]:
IMG_DIRECTORY = "data/images"
ARCH = "convnext" # to select the model to use, options: convnext, resnet50, efficientnet
IMAGE_SIZE = 224
BATCH = 8
keras.mixed_precision.set_global_policy('mixed_float16')
EPOCHS_HEAD = 5 # for feature extraction purposes
EPOCHS_FINE_TUNING = 20 # to fine tune
LR_HEAD = 1e-3
LR_FINE = 3e-5
SEED = 42

In [5]:
import pandas as pd
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Pandas: 3.0.5


In [6]:
!pip install pandas==3.0.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 23.0 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


In [21]:
keras.utils.set_random_seed(SEED)
train = pd.read_pickle("train.pkl")
validation = pd.read_pickle("validation.pkl")
test = pd.read_pickle("test.pkl")

In [22]:
for image in (train, validation, test):
    image['path'] = IMG_DIRECTORY + "/" + image['filename']

In [23]:
NUM_CLASSES = train['encoded_id'].nunique()
print(len(train), len(validation), len(test))

41991 8998 8999


Since the dataset is imbalanced:

In [24]:
class_weights = compute_class_weight("balanced", classes = np.arange(NUM_CLASSES), y = train['encoded_id'].values)

class_weights = tf.constant(class_weights, dtype = tf.float32)

In [25]:
def load_image(path):

    image = tf.io.decode_jpeg(tf.io.read_file(path), channels = 3) # since we have RBG images, there are 3 channels
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE), method = 'bicubic')
    return tf.clip_by_value(image, 0.0, 255.0)

In [26]:
def make_tf_dataset(dataframe, training):

    paths = dataframe['path'].values
    ids = dataframe['encoded_id'].values.astype('int32')

    dataset = tf.data.Dataset.from_tensor_slices((paths, ids))

    if training:
        dataset = dataset.shuffle(len(dataframe), seed = SEED, reshuffle_each_iteration = True)

    def _map(path, id): # the _ symbolises an internal function
        x = load_image(path)

        if training:
            return x, id, tf.gather(class_weights, id)
        return x, id

    return dataset.map(_map, num_parallel_calls = tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [27]:
train_ds = make_tf_dataset(train, True)
val_ds = make_tf_dataset(validation, False)
test_ds = make_tf_dataset(test, False)

In [28]:
def build_model():
    shape = (IMAGE_SIZE, IMAGE_SIZE, 3)

    if ARCH == "convnext":

        base = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet", input_shape=shape)
        prep = layers.Identity()

    elif ARCH == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=shape)
        prep = layers.Lambda(keras.applications.resnet50.preprocess_input)

    elif ARCH == "efficientnet":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=shape)
        prep = layers.Identity()

    else:
        raise ValueError(ARCH)

    input = keras.Input(shape)
    x = prep(input)
    x = base(x, training = False)

    x = layers.GlobalAveragePooling2D()(x)
    #x = layers.Dropout(0.3)(x)

    output = layers.Dense(NUM_CLASSES, activation = 'softmax', dtype = 'float32')(x)

    return keras.Model(input, output), base

In [14]:
class MacroF1(keras.metrics.Metric):
    def __init__(self, num_classes = NUM_CLASSES, name = 'macro_f1', **kwargs):
        super().__init__(name = name, **kwargs)

        self.num_classes = num_classes
        self.cm = self.add_weight(
            name = 'confusion_matrix',
            shape = (num_classes, num_classes),
            initializer = 'zeros'
        )
    def update_state(self, y_true, y_pred, sample_weight = None):
        y_true = keras.ops.cast(keras.ops.reshape(y_true, (-1,)), 'int32')
        y_pred = keras.ops.argmax(y_pred, axis = -1)

        indices = y_true * self.num_classes + y_pred

        cm = keras.ops.reshape(
            keras.ops.bincount(
                indices,
                minlength = self.num_classes ** 2
            ),
            (self.num_classes, self.num_classes)
        )

        self.cm.assign_add(keras.ops.cast(cm, self.dtype))

    def result(self):

        true_positives = keras.ops.diag(self.cm)
        false_positives = keras.ops.sum(self.cm, axis = 0) - true_positives
        false_negatives = keras.ops.sum(self.cm, axis = 1) - true_positives

        precision = true_positives / (true_positives + false_positives + keras.backend.epsilon())
        recall = true_positives / (true_positives + false_negatives + keras.backend.epsilon())

        f1_score = 2 * precision * recall / (
            precision + recall + keras.backend.epsilon()
        )

        return keras.ops.mean(f1_score)

    def reset_state(self):
        self.cm.assign(keras.ops.zeros_like(self.cm))

In [29]:
model, base = build_model()

loss_function = tf.keras.losses.SparseCategoricalCrossentropy()

metrics_list = [MacroF1(num_classes = NUM_CLASSES),
                keras.metrics.SparseTopKCategoricalAccuracy(5, name = "top5")]

In [17]:
base.trainable = False

model.compile(optimizer = keras.optimizers.SGD(LR_HEAD), loss = loss_function, metrics = metrics_list, jit_compile = False)

model.fit(train_ds, validation_data = val_ds, epochs = EPOCHS_HEAD)

Epoch 1/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 722s 1s/step - loss: 5.4796 - macro_f1: 0.0024 - top5: 0.0222 - val_loss: 5.4214 - val_macro_f1: 0.0030 - val_top5: 0.0258
Epoch 2/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 738s 1s/step - loss: 5.3530 - macro_f1: 0.0049 - top5: 0.0326 - val_loss: 5.3078 - val_macro_f1: 0.0071 - val_top5: 0.0382
Epoch 3/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 713s 1s/step - loss: 5.2619 - macro_f1: 0.0093 - top5: 0.0488 - val_loss: 5.2182 - val_macro_f1: 0.0124 - val_top5: 0.0600
Epoch 4/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 713s 1s/step - loss: 5.1848 - macro_f1: 0.0164 - top5: 0.0777 - val_loss: 5.1403 - val_macro_f1: 0.0211 - val_top5: 0.0934
Epoch 5/5
657/657 ━━━━━━━━━━━━━━━━━━━━ 710s 1s/step - loss: 5.1140 - macro_f1: 0.0287 - top5: 0.1130 - val_loss: 5.0679 - val_macro_f1: 0.0357 - val_top5: 0.1280


In [30]:
base.trainable = True
steps = EPOCHS_FINE_TUNING * len(train_ds)

lr = keras.optimizers.schedules.CosineDecay(LR_FINE, decay_steps=steps)

model.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay = 1e-4), loss = loss_function, metrics = metrics_list, jit_compile = False)

callbacks = [
    keras.callbacks.EarlyStopping(monitor = "val_loss", patience = 4, restore_best_weights = True), # THIS HAS TO BE CHANGED TO val_f1
    keras.callbacks.ModelCheckpoint(f"best_{ARCH}.keras", monitor = "val_loss", save_best_only = True), # THIS HAS TO BE CHANGED TO val_f1
]
model.fit(train_ds, validation_data = val_ds, epochs = EPOCHS_FINE_TUNING, callbacks = callbacks)

Epoch 1/20
5249/5249 ━━━━━━━━━━━━━━━━━━━━ 1236s 225ms/step - loss: 3.4908 - macro_f1: 0.3472 - top5: 0.5628 - val_loss: 2.1752 - val_macro_f1: 0.5606 - val_top5: 0.7714
Epoch 2/20
5249/5249 ━━━━━━━━━━━━━━━━━━━━ 1209s 230ms/step - loss: 1.8445 - macro_f1: 0.6619 - top5: 0.8290 - val_loss: 1.9247 - val_macro_f1: 0.6127 - val_top5: 0.8043
Epoch 3/20
5249/5249 ━━━━━━━━━━━━━━━━━━━━ 1173s 223ms/step - loss: 1.2606 - macro_f1: 0.7706 - top5: 0.8834 - val_loss: 1.8438 - val_macro_f1: 0.6335 - val_top5: 0.8128
Epoch 4/20
5249/5249 ━━━━━━━━━━━━━━━━━━━━ 1159s 221ms/step - loss: 0.8215 - macro_f1: 0.8490 - top5: 0.9218 - val_loss: 1.8597 - val_macro_f1: 0.6392 - val_top5: 0.8122
Epoch 5/20
5249/5249 ━━━━━━━━━━━━━━━━━━━━ 1159s 221ms/step - loss: 0.4922 - macro_f1: 0.9100 - top5: 0.9537 - val_loss: 1.9244 - val_macro_f1: 0.6377 - val_top5: 0.8040
Epoch 6/20
5249/5249 ━━━━━━━━━━━━━━━━━━━━ 1150s 219ms/step - loss: 0.2719 - macro_f1: 0.9488 - top5: 0.9760 - val_loss: 1.9874 - val_macro_f1: 0.6345 - val

In [31]:
y_true = test["encoded_id"].values

y_pred = model.predict(test_ds).argmax(axis = 1)

print("Test accuracy:", (y_true == y_pred).mean())
print("Test macro-F1:", f1_score(y_true, y_pred, average = "macro"))

names = train.drop_duplicates("encoded_id").set_index("encoded_id")["description"]

print(classification_report(y_true, y_pred, target_names=[names[i][:25] for i in range(NUM_CLASSES)]))

1125/1125 ━━━━━━━━━━━━━━━━━━━━ 50s 42ms/step
Test accuracy: 0.6768529836648517
Test macro-F1: 0.6366434251533216
                           precision    recall  f1-score   support

goldfish, Carassius aurat       0.94      0.85      0.89        52
European fire salamander,       0.69      0.53      0.60        17
bullfrog, Rana catesbeian       0.73      0.72      0.73        80
tailed frog, bell toad, r       0.72      0.55      0.63        76
American alligator, Allig       0.75      0.50      0.60        30
boa constrictor, Constric       0.61      0.77      0.68        78
                trilobite       0.62      0.59      0.61        17
                 scorpion       0.40      0.59      0.48        17
black widow, Latrodectus        0.55      0.67      0.60        18
                tarantula       0.83      0.80      0.81        30
                centipede       0.86      0.63      0.73        78
                    goose       0.67      0.63      0.65        19
koala, koala be